# Programme 6 — Interpolation et phénomène de Runge

**Référence :** Kröger, *Numerische Mathematik 1 für AMP*, chapitre 4.

Ce notebook accompagne `runge_phenomenon.py` et illustre :

1. Les trois méthodes : **Lagrange** (4.2), **Newton** (Satz 4.8), **Neville-Aitken** (Lemma 4.6)
2. La reproduction des **Übungen 4.5 et 4.7** du script
3. La borne d'erreur d'interpolation (**Satz 4.10**) et le rôle du **polynôme nodal**
4. Le **phénomène de Runge** : oscillations catastrophiques aux bords
5. La solution : **nœuds de Tchebychev**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from runge_phenomenon import (
    interpolation_lagrange, interpolation_newton, neville_aitken,
    differences_divisees, polynome_nodal,
    noeuds_equidistants, noeuds_tchebychev, runge,
    tracer_runge_equidistant, tracer_runge_tchebychev,
    tracer_polynome_nodal, tracer_erreur_vs_degre,
)

## 1. Existenz und Eindeutigkeit (Satz 4.2)

Pour $n+1$ nœuds distincts et $n+1$ valeurs, il existe **un unique** polynôme de degré $\leq n$ passant par tous les points. Trois algorithmes distincts pour le calculer, trois avantages complémentaires :

| Méthode | Coût construction | Coût par évaluation | Avantage |
|---|---|---|---|
| Lagrange | aucun | $O(n^2)$ | formule fermée, utile en théorie |
| Newton | $O(n^2)$ (diff. divisées) | $O(n)$ (Horner) | le plus rapide en évaluation |
| Neville-Aitken | aucun | $O(n^2)$ | pas besoin des coefficients |

## 2. Übung 4.5 — Lagrange et Newton sur 3 points

$x_0=1, x_1=2, x_2=4$ avec $y_0=6, y_1=6, y_2=0$.

In [ ]:
xs = np.array([1, 2, 4], dtype=float)
ys = np.array([6, 6, 0], dtype=float)

c = differences_divisees(xs, ys)
print(f'Différences divisées : c = {c}')
print(f'→ p(x) = {c[0]} + {c[1]}(x-{xs[0]}) + ({c[2]})(x-{xs[0]})(x-{xs[1]})')
print(f'       = -x² + 3x + 4')

x_fine = np.linspace(0, 5, 200)
p_lag = interpolation_lagrange(xs, ys, x_fine)
p_new = interpolation_newton(xs, ys, x_fine)

print(f'\n||Lagrange - Newton||_∞ = {np.max(np.abs(p_lag - p_new)):.2e}  (identiques ✓)')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x_fine, p_new, 'b-', linewidth=2, label='$p(x) = -x^2 + 3x + 4$')
plt.plot(xs, ys, 'ro', markersize=10, label='données')
plt.xlabel('$x$'); plt.ylabel('$p(x)$')
plt.title('Übung 4.5 — Interpolation de Lagrange / Newton')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 3. Übung 4.7 — Neville-Aitken

$x_0=0, x_1=1, x_2=3, x_3=4$, $y_0=12, y_1=3, y_2=-3, y_3=12$. Évaluer $p(2)$.

In [ ]:
xs2 = np.array([0, 1, 3, 4], dtype=float)
ys2 = np.array([12, 3, -3, 12], dtype=float)

val = neville_aitken(xs2, ys2, 2.0)
print(f'p(2) = {val}  (via Neville-Aitken)')
print(f'p(2) = {interpolation_newton(xs2, ys2, np.array([2.0]))[0]}  (via Newton, vérification)')

## 4. Le phénomène de Runge

La fonction de Runge $f(x) = \frac{1}{1 + 25x^2}$ est lisse, simple, symétrique. Pourtant, si on l'interpole sur $[-1, 1]$ avec des nœuds **équidistants** et qu'on augmente le degré $n$, l'erreur **augmente** au lieu de diminuer !

Les oscillations explosent aux bords de l'intervalle.

In [ ]:
tracer_runge_equidistant(degres=(5, 10, 15, 20))
plt.show()

Pour $n = 20$, les oscillations atteignent des amplitudes supérieures à **1000** près des bords — alors que la fonction elle-même reste dans $[0, 1]$ !

## 5. Pourquoi ? Le polynôme nodal (Satz 4.10)

La borne d'erreur dit :
$$|p(x) - f(x)| \leq \frac{1}{(n+1)!} \cdot \underbrace{\left|\prod_{i=0}^n (x - x_i)\right|}_{\text{polynôme nodal } |\omega(x)|} \cdot \max_{[a,b]} |f^{(n+1)}|$$

Avec des nœuds équidistants, $|\omega(x)|$ explose aux bords de l'intervalle. C'est **le** facteur responsable du phénomène de Runge.

In [ ]:
tracer_polynome_nodal(n=15)
plt.show()

Le polynôme nodal équidistant (rouge) est **~100× plus grand** aux bords que celui de Tchebychev (bleu). C'est cette différence qui fait tout.

## 6. La solution : nœuds de Tchebychev

$$x_k = \frac{a+b}{2} + \frac{b-a}{2} \cos\left(\frac{(2k+1)\pi}{2(n+1)}\right), \quad k = 0, \dots, n$$

Les nœuds sont concentrés aux bords (là où ça compte), et $|\omega(x)|$ est minimisé uniformément.

Résultat : l'interpolation **converge** quand $n$ augmente.

In [ ]:
tracer_runge_tchebychev(degres=(5, 10, 15, 20))
plt.show()

Plus aucune oscillation. À $n = 20$, l'approximation est visuellement indistinguable de la fonction.

## 7. Erreur maximale en fonction du degré

In [ ]:
tracer_erreur_vs_degre()
plt.show()

**Lecture :**

- **Équidistant (rouge) :** l'erreur **croît exponentiellement** avec $n$. À $n = 30$, on est à $\sim 10^4$. Augmenter le nombre de points aggrave les choses !
- **Tchebychev (bleu) :** l'erreur **décroît exponentiellement** avec $n$. À $n = 30$, on est à $\sim 10^{-8}$. Plus on ajoute de points, mieux c'est.

C'est un résultat à 12 ordres de grandeur de différence pour le même degré polynomial.

## 8. Comparaison from-scratch ↔ NumPy

In [ ]:
n = 10
xs = noeuds_tchebychev(-1, 1, n)
ys = runge(xs)
x_test = np.linspace(-1, 1, 200)

p_mine = interpolation_newton(xs, ys, x_test)
coeffs = np.polyfit(xs, ys, n)
p_numpy = np.polyval(coeffs, x_test)

print(f'||mine - numpy||_∞ = {np.max(np.abs(p_mine - p_numpy)):.2e}')

## Récapitulatif

| Point clé | Référence |
|---|---|
| Unicité du polynôme interpolant | Satz 4.2 |
| Lagrange : formule fermée mais $O(n^2)$ | Section 4.2.2 |
| Newton + Horner : $O(n^2)$ init, $O(n)$ par évaluation | Satz 4.8 |
| Borne d'erreur dépend du polynôme nodal $\omega(x)$ | Satz 4.10 |
| Nœuds équidistants → Runge (diverge !) | Section 4.2.6 |
| Nœuds de Tchebychev → convergence exponentielle | Section 4.2.6 |

**Morale :** en interpolation polynomiale, le choix des nœuds est **plus important** que le degré du polynôme. C'est une leçon fondamentale de l'analyse numérique.

---

*C'était le dernier des 6 programmes ! Les 21 programmes restants (voir l'inventaire complet) peuvent être implémentés de la même façon, chapitre par chapitre.*